# Foundations III: Limitations and pitfalls of ML systems

Companion to `slides/slides_01.md`.

* **D5** Under/overfitting and the bias–variance trade-off.
* **D6** The curse of dimensionality.
* **D7** No Free Lunch: no model wins everywhere.
* **D8** Data leakage: learning from pure noise.
* **D9** Correlation is not causation.
* **D10** Spurious correlations (shortcuts) under distribution shift.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from numpy.polynomial import Polynomial
from sklearn.datasets import make_circles, make_classification
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
rng = np.random.default_rng(SEED)
plt.rcParams["figure.figsize"] = (9, 4)

## D5 — Under/overfitting and the bias–variance trade-off

Imagine we could collect **30 different training sets** from the same process $y = \sin(2\pi x) + \varepsilon$: the same 20 positions $x$, but fresh noise each time.
We fit the same model to each one.

* **Bias:** how far the *average* fitted curve is from the truth. It is a systematic error: the model is too rigid.
* **Variance:** how much the fitted curves *disagree* with each other. The model is too sensitive to the particular sample.

$$\text{expected error} = \text{bias}^2 + \text{variance} + \text{noise}$$

In [ ]:
SIGMA, N_POINTS, N_DATASETS = 0.3, 20, 30
grid = np.linspace(0, 1, 200)
truth = np.sin(2 * np.pi * grid)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
print(f"{'degree':>8s}{'bias^2':>10s}{'variance':>10s}")
for ax, degree, label in zip(axes, [1, 5, 15], ["underfitting (high bias)", "good fit", "overfitting (high variance)"]):
    fits = []
    x = np.linspace(0, 1, N_POINTS)
    for _ in range(N_DATASETS):
        y = np.sin(2 * np.pi * x) + rng.normal(0, SIGMA, N_POINTS)
        fits.append(Polynomial.fit(x, y, deg=degree)(grid))
    fits = np.array(fits)
    bias2 = np.mean((fits.mean(axis=0) - truth) ** 2)
    variance = np.mean(fits.var(axis=0))
    print(f"{degree:8d}{bias2:10.3f}{variance:10.3f}")

    for f in fits:
        ax.plot(grid, f, color="tab:blue", alpha=0.15)
    ax.plot(grid, fits.mean(axis=0), color="tab:red", lw=2, label="average fit")
    ax.plot(grid, truth, "k--", lw=2, label="truth")
    ax.set(title=f"degree {degree}: {label}", ylim=(-2, 2), xlabel="x")
axes[0].legend()
plt.tight_layout()
plt.show()

* **Degree 1:** all 30 lines agree with each other (low variance) but are all wrong in the same way (high bias).
* **Degree 15:** on average the curves follow the truth (low bias), but each one wiggles differently (high variance).
* **Degree 5:** a balance. More complexity trades bias for variance; the best model minimizes their sum.

## D6 — The curse of dimensionality

### Distances lose their meaning

Draw 500 random points in the unit hypercube $[0,1]^d$ and measure, from a query point, how far the **nearest** and **farthest** points are.

$$\text{relative contrast} = \frac{d_{\max} - d_{\min}}{d_{\min}}$$

In [ ]:
dims = [1, 2, 5, 10, 20, 50, 100, 500, 1000]
contrast, ratio = [], []
for d in dims:
    points = rng.uniform(size=(500, d))
    query = rng.uniform(size=d)
    dist = np.linalg.norm(points - query, axis=1)
    contrast.append((dist.max() - dist.min()) / dist.min())
    ratio.append(dist.min() / dist.max())

for d, c, r in zip(dims, contrast, ratio):
    print(f"d = {d:5d}   relative contrast = {c:8.3f}   nearest/farthest = {r:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].loglog(dims, contrast, "o-")
axes[0].set(
    xlabel="dimension d", ylabel="relative contrast", title="Nearest and farthest neighbours become equally far"
)
volume_shell = [1 - 0.9**d for d in dims]
axes[1].semilogx(dims, volume_shell, "o-", color="tab:red")
axes[1].set(xlabel="dimension d", ylabel="fraction of volume", title="Volume within 5% of the cube's border")
plt.tight_layout()
plt.show()

In high dimensions almost all points are close to the border, and "near" and "far" become indistinguishable.

### Irrelevant features hurt distance-based models

Keep 2 informative features and add more and more pure-noise features.

In [ ]:
X_inf, y_inf = make_classification(
    n_samples=400, n_features=2, n_informative=2, n_redundant=0, class_sep=1.5, random_state=SEED
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
noise_dims = [0, 5, 10, 50, 100, 500, 1000]
knn_acc = []
for extra in noise_dims:
    X_noisy = np.hstack([X_inf, rng.normal(size=(len(X_inf), extra))])
    knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
    knn_acc.append(cross_val_score(knn, X_noisy, y_inf, cv=cv).mean())
    print(f"2 informative + {extra:4d} noise features -> k-NN accuracy = {knn_acc[-1]:.3f}")

The signal is still there, but it is buried: more features is **not** automatically better. With a fixed amount of data,
the number of samples needed to cover the space grows **exponentially** with dimension.

## D7 — No Free Lunch

Averaged over *all* possible problems, every learning algorithm performs the same. A model is good only when its **assumptions match the problem**.

In [ ]:
datasets = {
    "concentric circles": make_circles(n_samples=400, noise=0.1, factor=0.4, random_state=SEED),
    "linear, 10 of 50 useful": make_classification(
        n_samples=200, n_features=50, n_informative=10, n_redundant=0, n_clusters_per_class=1, random_state=SEED
    ),
}
models = {
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "k-NN (k=5)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
}
print(f"{'':24s}" + "".join(f"{m:>22s}" for m in models))
for dname, (Xd, yd) in datasets.items():
    row = [cross_val_score(m, Xd, yd, cv=cv).mean() for m in models.values()]
    print(f"{dname:24s}" + "".join(f"{a:22.3f}" for a in row))

The ranking **flips** between problems. There is no universally best model, only models whose inductive bias fits the data.

## D8 — Data leakage: 90% accuracy on pure noise

100 samples, 10 000 features, and **random** labels. There is nothing to learn: the honest accuracy is 50%.

A common mistake is to select the "best" features using **all** the data, and only afterwards run cross-validation.

In [ ]:
X_noise = rng.normal(size=(100, 10_000))
y_noise = rng.integers(0, 2, size=100)
clf = LogisticRegression(max_iter=1000)

# Wrong: feature selection sees the test folds
X_selected = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
leaky = cross_val_score(clf, X_selected, y_noise, cv=cv).mean()

# Right: feature selection is part of the model, refit inside each training fold
honest = cross_val_score(make_pipeline(SelectKBest(f_classif, k=20), clf), X_noise, y_noise, cv=cv).mean()

print(f"Feature selection BEFORE cross-validation: accuracy = {leaky:.3f}   <- leakage")
print(f"Feature selection INSIDE cross-validation: accuracy = {honest:.3f}   <- honest")

**Rule:** any step that learns from data (scaling, feature selection, imputation, tuning) must be fitted on the training part only.
Other common forms of leakage: duplicated samples across splits, features that encode the target (e.g. "treatment given"),
and using the future to predict the past in time series.

## D9 — Correlation is not causation

A beach town records, for 365 days, the **temperature**, the **ice-cream sales**, and the number of **drowning incidents**.

The true mechanism (which the model never sees):

* hot days → more ice cream is sold;
* hot days → more people swim → more drownings;
* ice cream has **no effect** on drownings.

In [ ]:
days = 365
temperature = rng.uniform(10, 35, days)
ice_cream = 20 * temperature + rng.normal(0, 60, days)


def drownings_given(temp):
    return np.maximum(0, 0.3 * temp - 3 + rng.normal(0, 1, len(temp)))


drownings = drownings_given(temperature)

print(f"Correlation(ice cream, drownings) = {np.corrcoef(ice_cream, drownings)[0, 1]:.2f}")

model_ice = LinearRegression().fit(ice_cream[:, None], drownings)
print(
    f"Model 'drownings ~ ice cream': R^2 = {model_ice.score(ice_cream[:, None], drownings):.2f}  <- a good predictor!"
)

The model predicts drownings well from ice-cream sales. Does that mean **restricting ice cream** would save lives?

An **intervention** changes the world: we force sales to **half**, while the weather stays the same.

In [ ]:
predicted_after = model_ice.predict(ice_cream[:, None] / 2).sum()
actual_after = drownings_given(temperature).sum()
print(f"Drownings per year before the intervention   : {drownings.sum():6.0f}")
print(f"Model's prediction with half the ice cream   : {predicted_after:6.0f}")
print(f"What actually happens with half the ice cream: {actual_after:6.0f}")

both = LinearRegression().fit(np.column_stack([ice_cream, temperature]), drownings)
print(
    f"\nAdding the confounder (temperature): weight of ice cream = {both.coef_[0]:+.4f}, weight of temperature = {both.coef_[1]:+.3f}"
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(ice_cream, drownings, s=8, alpha=0.6)
axes[0].set(xlabel="ice-cream sales", ylabel="drownings", title="Strong correlation...")
sc = axes[1].scatter(ice_cream, drownings, c=temperature, cmap="coolwarm", s=8)
fig.colorbar(sc, ax=axes[1], label="temperature (°C)")
axes[1].set(xlabel="ice-cream sales", ylabel="drownings", title="...explained by a hidden common cause")
plt.tight_layout()
plt.show()

A model trained on **observational** data learns associations. It predicts well as long as the world does not change,
but it cannot tell what will happen when we **act** on a feature. Once the confounder is included, the weight of ice cream drops to about zero.

## D10 — Spurious correlations and distribution shift

A classifier for a two-class problem with two features:

* `core`: truly causes the label, but is **noisy**.
* `shortcut`: for example, the background of a photo or the hospital that took the X-ray. In the **training** data it agrees with the label 95% of the time; in **deployment** that relationship is gone.

In [ ]:
def make_data(n, shortcut_agreement):
    labels = rng.integers(0, 2, size=n)
    core = (2 * labels - 1) + rng.normal(0, 1.5, n)
    agrees = rng.uniform(size=n) < shortcut_agreement
    shortcut_sign = np.where(agrees, 2 * labels - 1, -(2 * labels - 1))
    shortcut = shortcut_sign + rng.normal(0, 0.3, n)
    return np.column_stack([core, shortcut]), labels


X_tr, y_tr = make_data(2000, shortcut_agreement=0.95)
X_iid, y_iid = make_data(2000, shortcut_agreement=0.95)
X_shift, y_shift = make_data(2000, shortcut_agreement=0.5)

model = LogisticRegression().fit(X_tr, y_tr)
core_only = LogisticRegression().fit(X_tr[:, :1], y_tr)

print(f"Weights learned: core = {model.coef_[0, 0]:.2f}, shortcut = {model.coef_[0, 1]:.2f}\n")
print(f"{'':30s}{'same distribution':>20s}{'after shift':>14s}")
print(f"{'model using both features':30s}{model.score(X_iid, y_iid):20.3f}{model.score(X_shift, y_shift):14.3f}")
print(
    f"{'model using core feature only':30s}{core_only.score(X_iid[:, :1], y_iid):20.3f}{core_only.score(X_shift[:, :1], y_shift):14.3f}"
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True, sharey=True)
for ax, (Xd, yd, title) in zip(
    axes, [(X_tr, y_tr, "training (shortcut works)"), (X_shift, y_shift, "deployment (shortcut broken)")]
):
    ax.scatter(Xd[:, 0], Xd[:, 1], c=yd, cmap="coolwarm", s=6, alpha=0.6)
    xs = np.linspace(-6, 6, 10)
    ax.plot(xs, -(model.coef_[0, 0] * xs + model.intercept_[0]) / model.coef_[0, 1], "k--", label="decision boundary")
    ax.set(title=title, xlabel="core feature", ylim=(-2.5, 2.5))
axes[0].set_ylabel("shortcut feature")
axes[0].legend()
plt.tight_layout()
plt.show()

The model did exactly what it was asked to do: minimize training error. The **shortcut** was the easiest pattern to compress.
A validation set drawn from the same distribution **cannot detect** this failure; only data from the deployment conditions can.

**Takeaways**

1. Too simple means high bias; too complex means high variance. Aim for the balance.
2. More dimensions need exponentially more data, and irrelevant features are not free.
3. No model is best everywhere (No Free Lunch).
4. Leakage produces great numbers from nothing; keep every learned step inside the validation loop.
5. Models learn correlations, not causes: predictions do not tell you what happens when you intervene.
6. When the distribution shifts, shortcuts break.